# **Laboratorio 2 — California Housing Regression**
### Importaciones

In [ ]:

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device)


## **Preprocesamiento del Dataset**

In [ ]:

data = fetch_california_housing()
X = data.data
y = data.target.reshape(-1,1)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# StandardScaler
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s  = scaler_X.transform(X_test)
y_train_s = scaler_y.fit_transform(y_train)
y_test_s  = scaler_y.transform(y_test)

def make_loader(X_arr, y_arr, batch_size=64, shuffle=True):
    X_t = torch.tensor(X_arr, dtype=torch.float32)
    y_t = torch.tensor(y_arr, dtype=torch.float32)
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


## **Definición de la Red Neuronal**

In [ ]:

class RegressionNN(nn.Module):
    def __init__(self, input_dim, hidden_sizes):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


## **Configuración de Experimentos**

In [ ]:

architectures = {"A":[16,8], "B":[64]}
loss_funcs = {"MSE": nn.MSELoss, "SmoothL1": nn.SmoothL1Loss}
optimizers = ["Adam", "SGD"]

epochs = 50
batch_size = 64
lr_adam = 1e-3
lr_sgd  = 1e-2
weight_decay = 0.0

os.makedirs("lab2_plots", exist_ok=True)


## **Entrenamiento y Evaluación de los 8 Modelos**

In [ ]:

results = []
loss_curves = {}

for arch_name, hidden in architectures.items():
    for loss_name, LossClass in loss_funcs.items():
        for opt_name in optimizers:
            run_name = f"{arch_name}_hid{hidden}_loss{loss_name}_opt{opt_name}"
            print("Running:", run_name)

            train_loader = make_loader(X_train_s, y_train_s, batch_size=batch_size, shuffle=True)
            test_loader  = make_loader(X_test_s,  y_test_s,  batch_size=batch_size, shuffle=False)

            model = RegressionNN(input_dim=X_train_s.shape[1], hidden_sizes=hidden).to(device)

            if opt_name == "Adam":
                optimizer = optim.Adam(model.parameters(), lr=lr_adam, weight_decay=weight_decay)
            else:
                optimizer = optim.SGD(model.parameters(), lr=lr_sgd, weight_decay=weight_decay)

            criterion = LossClass()
            epoch_losses = []

            for ep in range(epochs):
                model.train()
                total_loss = 0.0
                total_n = 0
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    out = model(xb)
                    loss = criterion(out, yb)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item() * xb.size(0)
                    total_n += xb.size(0)

                avg_loss = total_loss / total_n
                epoch_losses.append(avg_loss)
                if (ep+1) % 10 == 0 or ep == 0:
                    print(f"  ep {ep+1}/{epochs}, loss {avg_loss:.6f}")

            loss_curves[run_name] = epoch_losses

            # Evaluate
            model.eval()
            preds_s, trues_s = [], []
            with torch.no_grad():
                for xb, yb in test_loader:
                    out = model(xb.to(device)).cpu().numpy()
                    preds_s.append(out)
                    trues_s.append(yb.numpy())

            preds_s = np.vstack(preds_s); trues_s = np.vstack(trues_s)
            preds = scaler_y.inverse_transform(preds_s)
            trues = scaler_y.inverse_transform(trues_s)

            mse = mean_squared_error(trues, preds)
            r2  = r2_score(trues, preds)

            results.append({
                "run": run_name,
                "architecture": str(hidden),
                "loss_fn": loss_name,
                "optimizer": opt_name,
                "mse": float(mse),
                "r2": float(r2)
            })

            # Save loss plot
            plt.figure(figsize=(6,4))
            plt.plot(epoch_losses)
            plt.title(f"Loss curve: {run_name}")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(os.path.join("lab2_plots", f"{run_name}_loss.png"))
            plt.close()


## **Resumen de Resultados**

In [ ]:

df = pd.DataFrame(results).sort_values("mse")
df.to_csv("lab2_results_summary.csv", index=False)
df


## **Gráficas de Métricas (MSE y R²)**

In [ ]:

plt.figure(figsize=(10,4))
plt.bar(df['run'], df['mse'])
plt.xticks(rotation=45, ha='right')
plt.title("Test MSE by experiment")
plt.tight_layout()
plt.savefig("lab2_plots/mse_by_experiment.png")
plt.close()

plt.figure(figsize=(10,4))
plt.bar(df['run'], df['r2'])
plt.xticks(rotation=45, ha='right')
plt.title("Test R2 by experiment")
plt.tight_layout()
plt.savefig("lab2_plots/r2_by_experiment.png")
plt.close()


## **Gráfica Predicción vs Real del Mejor Modelo**

In [ ]:

best = df.iloc[0]
best_run = best['run']

hidden = eval(best['architecture'])
loss_name = best['loss_fn']
opt_name = best['optimizer']
LossClass = loss_funcs[loss_name]

torch.manual_seed(0); np.random.seed(0)
train_loader = make_loader(X_train_s, y_train_s, batch_size=batch_size, shuffle=True)
test_loader  = make_loader(X_test_s,  y_test_s,  batch_size=batch_size, shuffle=False)

model = RegressionNN(input_dim=X_train_s.shape[1], hidden_sizes=hidden).to(device)

if opt_name == "Adam":
    optimizer = optim.Adam(model.parameters(), lr=lr_adam, weight_decay=weight_decay)
else:
    optimizer = optim.SGD(model.parameters(), lr=lr_sgd, weight_decay=weight_decay)

# Retrain
for ep in range(epochs):
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = LossClass()(out, yb)
        loss.backward()
        optimizer.step()

# Eval
model.eval()
preds_s, trues_s = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb.to(device)).cpu().numpy()
        preds_s.append(out)
        trues_s.append(yb.numpy())

preds_s = np.vstack(preds_s); trues_s = np.vstack(trues_s)
preds = scaler_y.inverse_transform(preds_s)
trues = scaler_y.inverse_transform(trues_s)

plt.figure(figsize=(6,6))
plt.scatter(trues, preds, s=8)
mi = min(trues.min(), preds.min()); ma = max(trues.max(), preds.max())
plt.plot([mi, ma], [mi, ma])
plt.xlabel("True median house value")
plt.ylabel("Predicted median house value")
plt.title(f"Pred vs True — Best: {best_run}")
plt.tight_layout()
plt.savefig("lab2_plots/best_pred_vs_true.png")
plt.close()

"Done."
